In [1]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [3]:
# Set up telemetry span exporter.
otel_exporter = OTLPSpanExporter(endpoint="http://localhost:4317", insecure=True)
span_processor = BatchSpanProcessor(otel_exporter)

# Set up telemetry trace provider.
tracer_provider = TracerProvider(resource=Resource({"service.name": "autogen-test-agentchat"}))
tracer_provider.add_span_processor(span_processor)
trace.set_tracer_provider(tracer_provider)

# Instrument the OpenAI Python library
OpenAIInstrumentor().instrument()

# we will get reference this tracer later using its service name
# tracer = trace.get_tracer("autogen-test-agentchat")

In [4]:
rag_vector_memory = await rag_indexer_runner('/run/media/jun/Jun 6T/Code/TFM_BA_GPT/lib/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [5]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [6]:
model_client = load_model_client("azure")

In [7]:
# Construct the absolute path to project/lib/agents_config.yaml
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user_proxy, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)


In [8]:
# 1️⃣ Grab the existing writer agent
old_writer = agents["functional_spec_writer_agent"]

# 2️⃣ Wrap the sub-agents as tools
use_case_tool = AgentTool(agent=agents["use_case_agent"])
glossary_tool = AgentTool(agent=agents["glossary_agent"])

# 3️⃣ Collect the writer's current tools (if any)
existing_tools = getattr(old_writer, "_tools", []) or []

# 4️⃣ Combine tools: existing ones + new agent-tools
combined_tools = existing_tools + [use_case_tool, glossary_tool]

# 5️⃣ Extract the system message text (AutoGen stores it as a list of SystemMessage objects)
system_message_text = ""
if hasattr(old_writer, "_system_messages") and old_writer._system_messages:
    system_message_text = old_writer._system_messages[0].content  # first SystemMessage content

# 6️⃣ Rebuild functional_spec_writer_agent with combined tools
agents["functional_spec_writer_agent"] = AssistantAgent(
    name=old_writer._name,
    description=old_writer._description,
    model_client=old_writer._model_client,
    memory=old_writer._memory,
    tools=combined_tools,
    reflect_on_tool_use=old_writer._reflect_on_tool_use,
    system_message=system_message_text,
    model_client_stream=old_writer._model_client_stream,
)

print("✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.")


✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.


In [9]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.AzureOpenAIChatCompletionClient object at 0x7f5e35144440>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f5e35145010>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Brea

In [10]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:

--- Agent: summarizer_agent ---
Tools:
Tool Name: save_summary_to_markdown
Description: 
Saves the final confirmed summary to a Markdown file.

Args:
    content (str): The final summary text, formatted in Markdown.



--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Ret

In [11]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [12]:
# Order agents (workflow order), user_proxy last
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
    "use_case_agent",
    "glossary_agent",
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user_proxy]

In [13]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False
)

In [14]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [15]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [16]:
await Console(team.run_stream(task=task))

await runtime.stop()

await model_client.close()

/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Task Breakdown:
1. Gather structured requirements by interviewing the customer.
2. Summarize gathered information.
3. Write the functional specification document.
4. Check the functional specification for completeness.
5. Create process flow diagrams based on the specifications.
6. Review Mermaid code for diagrams for correctness and clarity.

**TASK LIST**
interviewer_agent: not started  
summarizer_agent: not started  
functional_spec_writer_agent: not started  
functional_spec_checker_agent: not started  
diagram_creator_agent: not started  
mermaid_code_reviewer_agent: not started  

**ASSIGNMENT**
1. interviewer_agent : Start gathering requirements from the user.


/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: interviewer_agent (attempt 1)
Selected speaker: interviewer_agent
DEBUG:autogen_agentchat:Selected speaker: interviewer_agent


---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='• \nFunctional Requirements Specification \n• \n1. Document Information  \no 1.1. Document Approvers & Reviewers \no 1.2. Document History \no 1.3. Reference Documents \no 1.4. Definitions, Acronyms and Abbreviations \n• \n2. Executive Summary  \no 2.1. Overview \no 2.2. Business Benefits \n• \n3. Scope  \no 3.1. In Scope \no 4.2. Out of Scope \no 3.3. Systems & Platforms LeanIX Meta-Data \n• \n4. Requirements  \no 4.1. Functional Requirements \no 4.2. Non-Functional Requirements \n• \n5. Functional Solution', mime_type='MemoryMimeType.TEXT', metadata={'chunk_index': 3, 'mime_type': 'MemoryMimeType.TEXT', 'source': 'Functional Requirements Specification Template.doc.pdf', 'score': 0.6246372759342194, 'id': '449a8a70-4043-4372-b176-0e7f32232870'}), MemoryContent(content='How Do You Ensure Accurate Requirements Gathering?Requirements analysts must know a host of methods to collect requirements from all the

/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected the previous speaker: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected the previous speaker: interviewer_agent (attempt 1)
Model selected a valid name: user_proxy (attempt 2)
DEBUG:autogen_agentchat:Model selected a valid name: user_proxy (attempt 2)
Selected speaker: user_proxy
DEBUG:autogen_agentchat:Selected speaker: user_proxy


---------- TextMessage (user_proxy) ----------
The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call


... (38 outputs left out)

id='860b07a2-4087-4d15-a657-b9d0036ad191' source='functional_spec_writer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 2, 22, 16, 22, 91076, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Markdown spec successfully written to ../conversation/functional_specification.md', name='write_md_file_tool', call_id='call_O2gkQDuDh6i8W8r50Gz0R3fZ', is_error=False)] type='ToolCallExecutionEvent'
id='860b07a2-4087-4d15-a657-b9d0036ad191' source='functional_spec_writer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 2, 22, 16, 22, 91076, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Markdown spec successfully written to ../conversation/functional_specification.md', name='write_md_file_tool', call_id='call_O2gkQDuDh6i8W8r50Gz0R3fZ', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='860b07a2-4087-4d15-a657-b9d0036ad191' source='functional_spec_writer_agent' models_

---------- ToolCallExecutionEvent (functional_spec_writer_agent) ----------
[FunctionExecutionResult(content='Markdown spec successfully written to ../conversation/functional_specification.md', name='write_md_file_tool', call_id='call_O2gkQDuDh6i8W8r50Gz0R3fZ', is_error=False)]
---------- ModelClientStreamingChunkEvent (functional_spec_writer_agent) ----------
The Functional Specification Document has been successfully saved. The document now includes all necessary sections and details. You can proceed with the next steps for review and validation. **READY**


ERROR:autogen_core:Error processing publish message for SelectorGroupChatManager_12272704-2735-4587-8816-255fc365c2ed/12272704-2735-4587-8816-255fc365c2ed
Traceback (most recent call last):
  File "/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/run/media/jun/Jun 6T/Co

RuntimeError: RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2024-12-01-preview have exceeded token rate limit of your current AIServices S0 pricing tier. Please retry after 14 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
Traceback:
Traceback (most recent call last):

  File "/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_base_group_chat_manager.py", line 164, in handle_agent_response
    await self._transition_to_next_speakers(ctx.cancellation_token)

  File "/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_base_group_chat_manager.py", line 176, in _transition_to_next_speakers
    speaker_names =